# Procurement Intelligence - Data Exploration

This notebook provides comprehensive data exploration and analysis capabilities for procurement data.

## Contents:
1. Data Loading and Initial Exploration
2. Data Quality Assessment
3. Descriptive Statistics
4. Time Series Analysis
5. Supplier Analysis
6. Department Analysis
7. Category Analysis
8. Correlation Analysis

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Import custom modules
import sys
sys.path.append('../src')
from data_processor import ProcurementDataProcessor
from analytics import ProcurementAnalytics
from visualizations import ProcurementVisualizations

# Set display options
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print("Libraries imported successfully!")

## 1. Data Loading and Initial Exploration

In [ ]:
# Initialize data processor
processor = ProcurementDataProcessor()

# Generate sample data (replace with actual data loading)
print("Generating sample procurement data...")
data = processor.generate_sample_data(num_records=2000)

# Clean the data
cleaned_data = processor.clean_data()

print(f"Dataset shape: {cleaned_data.shape}")
print(f"Date range: {cleaned_data['Date'].min()} to {cleaned_data['Date'].max()}")
print(f"Total records: {len(cleaned_data):,}")

In [ ]:
# Display first few rows
print("Sample Data:")
display(cleaned_data.head())

# Display data information
print("\nData Information:")
display(cleaned_data.info())

## 2. Data Quality Assessment

In [ ]:
# Check for missing values
print("Missing Values:")
missing_values = cleaned_data.isnull().sum()
missing_percentage = (missing_values / len(cleaned_data)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_values,
    'Missing Percentage': missing_percentage
})
display(missing_df[missing_df['Missing Count'] > 0])

# Check for duplicates
duplicates = cleaned_data.duplicated().sum()
print(f"\nDuplicate records: {duplicates}")

# Check data types
print("\nData Types:")
display(cleaned_data.dtypes)

## 3. Descriptive Statistics

In [ ]:
# Numeric columns statistics
print("Numeric Columns Statistics:")
numeric_stats = cleaned_data.describe().round(2)
display(numeric_stats)

# Categorical columns statistics
print("\nCategorical Columns Statistics:")
categorical_columns = cleaned_data.select_dtypes(include=['object']).columns
for col in categorical_columns:
    print(f"\n{col}:")
    print(f"  Unique values: {cleaned_data[col].nunique()}")
    print(f"  Top 5 values:")
    display(cleaned_data[col].value_counts().head())

## 4. Key Metrics Calculation

In [ ]:
# Calculate key metrics
metrics = processor.calculate_metrics()

print("Key Performance Indicators:")
for metric, value in metrics.items():
    if isinstance(value, float):
        print(f"  {metric.replace('_', ' ').title()}: {value:,.2f}")
    else:
        print(f"  {metric.replace('_', ' ').title()}: {value}")

## 5. Time Series Analysis

In [ ]:
# Monthly spending trend
cleaned_data['Month'] = cleaned_data['Date'].dt.to_period('M')
monthly_spend = cleaned_data.groupby('Month')['Amount'].sum().reset_index()
monthly_spend['Month'] = monthly_spend['Month'].astype(str)

# Create interactive plot
fig = px.line(monthly_spend, x='Month', y='Amount', 
              title='Monthly Spending Trend',
              markers=True,
              labels={'Amount': 'Total Spend ($)', 'Month': 'Month'})
fig.update_layout(height=500)
fig.show()

# Display monthly statistics
print("Monthly Spending Statistics:")
print(f"  Average monthly spend: ${monthly_spend['Amount'].mean():,.2f}")
print(f"  Peak month: {monthly_spend.loc[monthly_spend['Amount'].idxmax(), 'Month']}")
print(f"  Lowest month: {monthly_spend.loc[monthly_spend['Amount'].idxmin(), 'Month']}")

## 6. Supplier Analysis

In [ ]:
# Top suppliers analysis
top_suppliers = processor.get_top_suppliers(top_n=10)
print("Top 10 Suppliers by Total Spend:")
display(top_suppliers)

# Supplier performance visualization
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Total Spend', 'Order Count', 'Average Lead Time', 'Completion Rate')
)

fig.add_trace(
    go.Bar(x=top_suppliers.index, y=top_suppliers['Total_Spend'], name='Total Spend'),
    row=1, col=1
)

fig.add_trace(
    go.Bar(x=top_suppliers.index, y=top_suppliers['Order_Count'], name='Order Count'),
    row=1, col=2
)

fig.add_trace(
    go.Bar(x=top_suppliers.index, y=top_suppliers['Avg_Lead_Time'], name='Avg Lead Time'),
    row=2, col=1
)

# Calculate completion rate for suppliers
supplier_completion = cleaned_data.groupby('Supplier')['Status'].apply(
    lambda x: (x == 'Completed').mean() * 100
).round(2)

fig.add_trace(
    go.Bar(x=supplier_completion.index, y=supplier_completion.values, name='Completion Rate'),
    row=2, col=2
)

fig.update_layout(height=600, showlegend=False, title_text="Supplier Performance Analysis")
fig.show()

## 7. Department Analysis

In [ ]:
# Department-wise analysis
dept_analysis = processor.get_department_analysis()
print("Department-wise Analysis:")
display(dept_analysis)

# Create department spending pie chart
fig = px.pie(
    values=dept_analysis['Total_Spend'],
    names=dept_analysis.index,
    title='Spending Distribution by Department'
)
fig.show()

# Department vs Average Order Value
fig2 = px.bar(
    x=dept_analysis.index,
    y=dept_analysis['Avg_Order_Value'],
    title='Average Order Value by Department',
    labels={'Avg_Order_Value': 'Average Order Value ($)', 'x': 'Department'}
)
fig2.update_xaxes(tickangle=45)
fig2.show()

## 8. Category Analysis

In [ ]:
# Category trends
category_trends = processor.get_category_trends()
print("Category-wise Monthly Trends:")
display(category_trends.head())

# Category spending distribution
category_spend = cleaned_data.groupby('Category')['Amount'].sum().sort_values(ascending=False)

fig = px.bar(
    x=category_spend.index,
    y=category_spend.values,
    title='Total Spending by Category',
    labels={'y': 'Total Spend ($)', 'x': 'Category'}
)
fig.update_xaxes(tickangle=45)
fig.show()

# Category growth analysis
print("\nCategory Growth Analysis:")
for category in category_spend.index:
    category_data = cleaned_data[cleaned_data['Category'] == category]
    if len(category_data) > 1:
        recent_spend = category_data[category_data['Date'] >= category_data['Date'].max() - pd.Timedelta(days=90)]['Amount'].sum()
        historical_spend = category_data[category_data['Date'] < category_data['Date'].max() - pd.Timedelta(days=90)]['Amount'].sum()
        if historical_spend > 0:
            growth = ((recent_spend - historical_spend) / historical_spend) * 100
            print(f"  {category}: {growth:.1f}% growth")

## 9. Correlation Analysis

In [ ]:
# Select numeric columns for correlation
numeric_columns = ['Amount', 'Quantity', 'Unit_Price', 'Lead_Time']
correlation_data = cleaned_data[numeric_columns].corr()

# Create correlation heatmap
fig = px.imshow(
    correlation_data,
    text_auto=True,
    aspect="auto",
    title='Correlation Matrix'
)
fig.show()

print("Correlation Analysis:")
display(correlation_data.round(3))

# Find strong correlations
strong_correlations = []
for i in range(len(correlation_data.columns)):
    for j in range(i+1, len(correlation_data.columns)):
        corr_value = correlation_data.iloc[i, j]
        if abs(corr_value) > 0.5:  # Strong correlation threshold
            strong_correlations.append({
                'Variable 1': correlation_data.columns[i],
                'Variable 2': correlation_data.columns[j],
                'Correlation': corr_value
            })

if strong_correlations:
    print("\nStrong Correlations (|r| > 0.5):")
    for corr in strong_correlations:
        print(f"  {corr['Variable 1']} - {corr['Variable 2']}: {corr['Correlation']:.3f}")
else:
    print("\nNo strong correlations found (|r| > 0.5)")

## 10. Advanced Analytics

In [ ]:
# Initialize analytics engine
analytics = ProcurementAnalytics(cleaned_data)

# Generate comprehensive insights
print("Generating advanced analytics insights...")
insights = analytics.generate_insights_report()

# Display key insights
print("\n=== SPENDING FORECAST ===")
forecast = insights['spending_forecast']
print(f"Forecast method: {forecast['method']}")
print(f"Model R-squared: {forecast['model_performance']['r_squared']:.3f}")
print(f"Next month forecast: ${forecast['values'][0]:,.2f}")

print("\n=== TOP SUPPLIER PERFORMERS ===")
for supplier in insights['supplier_performance']['top_performers'][:3]:
    print(f"  • {supplier}")

print("\n=== OPTIMIZATION OPPORTUNITIES ===")
for opportunity in insights['optimization_opportunities']['opportunities']:
    print(f"  • {opportunity['type']}: {opportunity['description']}")

print("\n=== RISK ASSESSMENT ===")
print(f"Overall risk level: {insights['risk_assessment']['risk_level']}")
print(f"Total identified risks: {insights['risk_assessment']['total_risks']}")

if insights['risk_assessment']['high_priority_risks']:
    print("High priority risks:")
    for risk in insights['risk_assessment']['high_priority_risks']:
        print(f"  • {risk['type']}: {risk['description']}")

## 11. Export Results

In [ ]:
# Export processed data
output_dir = '../data/processed'
import os
os.makedirs(output_dir, exist_ok=True)

# Export cleaned data
cleaned_data.to_csv(f'{output_dir}/procurement_data_cleaned.csv', index=False)
print(f"Cleaned data exported to: {output_dir}/procurement_data_cleaned.csv")

# Export analysis results
analysis_results = {
    'metrics': metrics,
    'top_suppliers': top_suppliers.to_dict(),
    'department_analysis': dept_analysis.to_dict(),
    'category_trends': category_trends.to_dict(),
    'insights': insights
}

import json
with open(f'{output_dir}/analysis_results.json', 'w') as f:
    json.dump(analysis_results, f, indent=2, default=str)

print(f"Analysis results exported to: {output_dir}/analysis_results.json")
print("\nData exploration completed successfully!")